# FL-06 — Comprehensive Data Quality Investigation v3

بررسی یکپارچه تعارض برچسب‌ها و مقادیر منفی CIC-IDS2017 با ردیابی ردیف‌ها تا فایل خام.

## پروتکل ایمنی
- فقط رکوردهای Train و Validation که در خروجی‌های تأییدشده Audit v2 فهرست شده‌اند تحلیل می‌شوند.
- هشت فایل خام با اندازه و SHA-256 کنترل می‌شوند.
- ردیف‌های موردنیاز از CSV خام به‌صورت chunk خوانده می‌شوند.
- هیچ برچسب، ویژگی یا فایل منبعی تغییر نمی‌کند.
- هیچ آموزش، پیش‌بینی، تنظیم یا ارزیابی Test انجام نمی‌شود.
- در صورت هرگونه ناسازگاری، اجرا متوقف می‌شود.
- نتایج CSV-level هستند؛ داوری قطعی برچسب به PCAP، زمان‌بندی حمله و بازسازی Flow نیاز دارد.

## خروجی‌ها
گزارش Markdown/JSON، توزیع ترکیب برچسب‌ها، ماتریس برچسب-منبع-Split، جزئیات گروه‌های تعارض، مشخصات گروه 3,362 ردیفی، اطلاعات خام Flow/5-tuple/Timestamp، تحلیل مقادیر منفی، هم‌رخدادی ویژگی‌ها و وابستگی Duration/Rate.

در Colab تمام سلول‌ها را به‌ترتیب اجرا کنید. Secret با نام GITHUB_TOKEN و دسترسی Read لازم است. انتشار خروجی‌ها در آخر اختیاری است.

In [1]:
import os, json, csv, hashlib, base64, subprocess, shutil, platform
from pathlib import Path
import numpy as np, pandas as pd

REPO="AmirHosseinSharifian/Federated-Learning"
ROOT=Path("/content/FL06-data-quality-v3")
PROJECT=ROOT/"CIC-IDS2017"
DATA=PROJECT/"data"
AUDIT=PROJECT/"FL-06_OneShot_GroupAware_v1_3_FinalPaper"/"audit"
V2=AUDIT/"source_row_audit_v2_reproducibility"
OUT=AUDIT/"data_quality_investigation_v3"
try:
    from google.colab import userdata
    TOKEN=userdata.get("GITHUB_TOKEN")
except Exception as e:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets and enable notebook access.") from e
if not TOKEN: raise RuntimeError("GITHUB_TOKEN missing.")
AUTH=base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
CFG=f"http.https://github.com/.extraheader=AUTHORIZATION: basic {AUTH}"
ENV=os.environ.copy(); ENV["GIT_LFS_SKIP_SMUDGE"]="1"; ENV["GIT_TERMINAL_PROMPT"]="0"
def git(args,cwd=None,check=True):
    p=subprocess.run(["git","-c",CFG,*args],cwd=cwd,env=ENV,text=True,capture_output=True)
    if check and p.returncode:
        raise RuntimeError((p.stderr or p.stdout).replace(TOKEN,"[REDACTED]")[-3000:])
    return p
if subprocess.run(["git","lfs","version"],capture_output=True).returncode:
    subprocess.run(["apt-get","update","-qq"],check=True); subprocess.run(["apt-get","install","-y","-qq","git-lfs"],check=True)
subprocess.run(["git","lfs","install","--system"],check=True)
if ROOT.exists(): shutil.rmtree(ROOT)
git(["clone","--depth","1",f"https://github.com/{REPO}.git",str(ROOT)])
git(["lfs","install","--local"],cwd=ROOT)
git(["lfs","pull","--include=CIC-IDS2017/data/*.csv"],cwd=ROOT)
HEAD=git(["rev-parse","HEAD"],cwd=ROOT).stdout.strip()
OUT.mkdir(parents=True,exist_ok=True)
print("HEAD",HEAD,"Python",platform.python_version(),"pandas",pd.__version__)

HEAD 775f2191d3466af15cd3073949d8058a116bcb78 Python 3.13.18 pandas 2.2.3


In [2]:
# 1. Verify raw source and upstream v2 integrity.
with (AUDIT/"raw_file_manifest.csv").open("r",encoding="utf-8-sig",newline="") as f: manifest=list(csv.DictReader(f))
assert len(manifest)==8
verification=[]
for r in manifest:
    p=DATA/r["file"]; h=hashlib.sha256(); n=0
    with p.open("rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): n+=len(b); h.update(b)
    ok=n==int(r["size_bytes"]) and h.hexdigest()==r["sha256"].strip().lower()
    verification.append({"file":r["file"],"size_bytes":n,"sha256":h.hexdigest(),"status":"PASS" if ok else "FAIL"})
    if not ok: raise RuntimeError("Source checksum failure: "+r["file"])
split=json.loads((AUDIT/"dataset_split_manifest.json").read_text())
summary_v2=json.loads((V2/"audit_summary.json").read_text())
comparison=json.loads((V2/"audit_comparison.json").read_text())
assert comparison.get("all_comparisons_passed") is True
assert split["dataset_fingerprint"]==summary_v2["dataset_fingerprint"]
assert split["split_fingerprint"]==summary_v2["split_fingerprint"]
assert summary_v2["test_predictions_performed"] is False and summary_v2["test_metrics_computed"] is False
print("8/8 source checksums PASS; v1-v2 comparison PASS.")
print("Dataset fingerprint",split["dataset_fingerprint"])
print("Split fingerprint",split["split_fingerprint"])

8/8 source checksums PASS; v1-v2 comparison PASS.
Dataset fingerprint 831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58
Split fingerprint b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812


In [3]:
# 2. Load Train/Validation-only v2 extracts and validate schemas.
conf=pd.read_csv(V2/"source_conflict_records_77_features_train_validation.csv",low_memory=False)
groups=pd.read_csv(V2/"source_conflict_groups_train_validation.csv",low_memory=False)
neg=pd.read_csv(V2/"negative_values_records_train_validation.csv",low_memory=False)
neg_summary=pd.read_csv(V2/"negative_values_summary_train_validation.csv",low_memory=False)
keys=["source_file","source_file_row_index_zero_based"]
assert len(conf)==6342 and conf.feature_hash.nunique()==614
assert len(groups)==614 and int(groups.rows.sum())==6342
assert set(["Label","split",*keys]).issubset(conf.columns)
assert set(["feature","value","Label","split",*keys]).issubset(neg.columns)
meta={"clean_row_index","feature_hash","class_id","split","source_file","source_file_row_index_zero_based","raw_global_row_index_zero_based","Label"}
features=[c for c in conf.columns if c not in meta]
assert len(features)==77
print("Conflict groups/rows/features:",conf.feature_hash.nunique(),len(conf),len(features))
print("Negative occurrences:",len(neg),"features:",sorted(neg.feature.unique()))

Conflict groups/rows/features: 614 6342 77
Negative occurrences: 3204 features: ['Bwd Header Length', 'Flow Bytes/s', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow Packets/s', 'Fwd Header Length', 'Fwd IAT Min', 'min_seg_size_forward']


In [4]:
# 3. Recover only the selected source rows from original CSVs (chunked, bounded memory).
requests=pd.concat([conf[keys],neg[keys]],ignore_index=True).drop_duplicates()
requests[keys[1]]=requests[keys[1]].astype("int64")
wanted={k:set(g[keys[1]].astype(int)) for k,g in requests.groupby(keys[0])}
parts=[]
for j,r in enumerate(manifest,1):
    name=r["file"]; need=wanted.get(name,set())
    if not need: continue
    offset=0; got=[]
    print(f"[{j}/8] {name}: {len(need)} rows requested")
    for chunk in pd.read_csv(DATA/name,chunksize=100000,low_memory=False):
        chunk.columns=chunk.columns.astype(str).str.strip()
        hi=offset+len(chunk); idx=[x for x in need if offset<=x<hi]
        if idx:
            z=chunk.iloc[np.asarray(idx,dtype=np.int64)-offset].copy()
            z.insert(0,keys[1],np.asarray(idx,dtype=np.int64)); z.insert(0,keys[0],name); got.append(z)
        offset=hi
    if sum(map(len,got))!=len(need): raise RuntimeError(f"Missing raw rows in {name}")
    parts.extend(got)
raw=pd.concat(parts,ignore_index=True)
raw["Label"]=raw["Label"].astype("string").str.strip()
assert len(raw)==len(requests)
for src in [conf[keys+["Label"]],neg[keys+["Label"]]]:
    z=src.merge(raw[keys+["Label"]],on=keys,suffixes=("_audit","_raw"),validate="many_to_one")
    if not (z.Label_audit.astype(str).to_numpy()==z.Label_raw.astype(str).to_numpy()).all():
        raise RuntimeError("Raw label mismatch against v2 audit.")
print("Raw rows recovered:",len(raw),"; all labels match.")

[1/8] Monday-WorkingHours.pcap_ISCX.csv: 487 rows requested
[2/8] Tuesday-WorkingHours.pcap_ISCX.csv: 485 rows requested
[3/8] Wednesday-workingHours.pcap_ISCX.csv: 5474 rows requested
[4/8] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 243 rows requested
[5/8] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 860 rows requested
[6/8] Friday-WorkingHours-Morning.pcap_ISCX.csv: 215 rows requested
[7/8] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 102 rows requested
[8/8] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 1090 rows requested
Raw rows recovered: 8956 ; all labels match.


In [5]:
# 4. Complete label-conflict analysis and largest-group provenance.
conf["feature_hash"]=conf.feature_hash.astype(str)
g=(conf.groupby("feature_hash",sort=False).agg(rows=("Label","size"),n_labels=("Label","nunique"),
 labels=("Label",lambda x:" || ".join(sorted(set(map(str,x))))),
 source_files=("source_file",lambda x:" || ".join(sorted(set(map(str,x))))),
 n_source_files=("source_file","nunique"),n_splits=("split","nunique")).reset_index())
assert len(g)==614 and int(g.rows.sum())==6342 and g.n_labels.eq(1).sum()==0
label_combos=g.groupby("labels").agg(groups=("feature_hash","size"),rows=("rows","sum")).reset_index().sort_values("groups",ascending=False)
label_source=conf.groupby(["Label","source_file","split"]).size().rename("rows").reset_index()
label_source_matrix=pd.crosstab(conf.Label,conf.source_file)
label_split_matrix=pd.crosstab(conf.Label,conf.split)
largest_hash=g.loc[g.rows.idxmax(),"feature_hash"]
largest=conf[conf.feature_hash==largest_hash].copy()
assert len(largest)==3362 and largest[features].drop_duplicates().shape[0]==1
largest_profile=pd.DataFrame({"feature":features,
 "unique_values":[largest[c].nunique(dropna=False) for c in features],
 "min":[pd.to_numeric(largest[c],errors="coerce").min() for c in features],
 "max":[pd.to_numeric(largest[c],errors="coerce").max() for c in features]})
largest_profile["constant"]=largest_profile.unique_values.eq(1)
raw_conf=raw.merge(conf[keys+["feature_hash","split","Label"]],on=keys,how="inner",suffixes=("_raw","_audit"),validate="one_to_one")
assert len(raw_conf)==len(conf)
# Exact cross-check of all 77 retained model features against original source CSV rows.
raw_conf=raw_conf.merge(conf[keys+features],on=keys,how="inner",suffixes=("_raw","_audit"),validate="one_to_one")
for feature in features:
    a=pd.to_numeric(raw_conf[feature+"_raw"],errors="coerce").to_numpy(dtype=float)
    b=pd.to_numeric(raw_conf[feature+"_audit"],errors="coerce").to_numpy(dtype=float)
    if not np.array_equal(a,b,equal_nan=True):
        raise RuntimeError("Raw 77-feature mismatch in conflict record: "+feature)
print("All 77 features exactly match original raw CSV for every conflict record.")
meta_cols=[c for c in ["Timestamp","Flow ID","Source IP","Source Port","Destination IP","Destination Port","Protocol"] if c in raw_conf]
if "Timestamp" in raw_conf:
    raw_conf["timestamp_parsed"]=pd.to_datetime(raw_conf.Timestamp,errors="coerce")
    time_profile=raw_conf.groupby("feature_hash").agg(timestamp_count=("timestamp_parsed","count"),timestamp_min=("timestamp_parsed","min"),timestamp_max=("timestamp_parsed","max"),timestamp_unique=("timestamp_parsed","nunique")).reset_index()
else: time_profile=pd.DataFrame()
if all(c in raw_conf for c in ["Source IP","Source Port","Destination IP","Destination Port","Protocol"]):
    tc=["Source IP","Source Port","Destination IP","Destination Port","Protocol"]
    raw_conf["five_tuple"]=raw_conf[tc].astype(str).agg("|".join,axis=1)
    tuple_profile=raw_conf.groupby("feature_hash").five_tuple.nunique().rename("unique_five_tuples").reset_index()
else: tuple_profile=pd.DataFrame()
largest_meta=raw_conf[raw_conf.feature_hash==largest_hash][keys+["Label_audit","split"]+meta_cols].drop_duplicates()
print("Largest group labels",sorted(largest.Label.unique()))
print("Largest group source counts\n",largest.source_file.value_counts().to_string())
print("Raw metadata fields",meta_cols)
if not time_profile.empty: print(time_profile[time_profile.feature_hash==largest_hash].to_string(index=False))
if not tuple_profile.empty: print("Largest group unique 5-tuples",int(tuple_profile.loc[tuple_profile.feature_hash==largest_hash,"unique_five_tuples"].iloc[0]))

All 77 features exactly match original raw CSV for every conflict record.
Largest group labels ['BENIGN', 'DoS Hulk']
Largest group source counts
 source_file
Wednesday-workingHours.pcap_ISCX.csv                    3360
Tuesday-WorkingHours.pcap_ISCX.csv                         1
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv       1
Raw metadata fields []


In [6]:
# 5. Negative values: feature counts, unique rows, co-occurrence and raw-value cross-check.
neg["value"]=pd.to_numeric(neg.value,errors="coerce")
per_feature=neg.groupby("feature").agg(occurrences=("value","size"),min_negative=("value","min"),max_negative=("value","max"),distinct_values=("value","nunique")).reset_index()
per_feature["unique_source_rows"]=[neg.loc[neg.feature==f,keys].drop_duplicates().shape[0] for f in per_feature.feature]
per_feature=per_feature.sort_values("occurrences",ascending=False)
neg_wide=neg.assign(flag=1).pivot_table(index=keys,columns="feature",values="flag",aggfunc="max",fill_value=0)
cooccur=neg_wide.T.dot(neg_wide)
patterns=neg_wide.astype(int).astype(str).agg("".join,axis=1).value_counts().rename_axis("pattern").reset_index(name="rows")
neg_raw=raw.merge(neg[keys+["feature","value","Label","split"]],on=keys,how="inner",suffixes=("_raw","_audit"),validate="one_to_many")
for _,r in neg_raw.iterrows():
    rv=pd.to_numeric(pd.Series([r.get(r.feature)]),errors="coerce").iloc[0]
    if pd.isna(rv) or float(rv)!=float(r.value):
        raise RuntimeError(f"Negative raw-value mismatch: {r.source_file} row {r.source_file_row_index_zero_based} {r.feature}")
print("Negative feature-value occurrences",len(neg))
print("Unique affected source rows",neg[keys].drop_duplicates().shape[0])
print(per_feature.to_string(index=False))
print("All negative raw values exactly match v2 extract.")

Negative feature-value occurrences 3204
Unique affected source rows 2614
             feature  occurrences  min_negative  max_negative  distinct_values  unique_source_rows
        Flow IAT Min         2581 -1.400000e+01 -1.000000e+00               12                2581
       Flow Duration          110 -1.300000e+01 -1.000000e+00                5                 110
      Flow Packets/s          110 -2.000000e+06 -1.538462e+05                5                 110
       Flow IAT Mean          110 -1.300000e+01 -1.000000e+00                5                 110
        Flow IAT Max          110 -1.300000e+01 -1.000000e+00                5                 110
        Flow Bytes/s           82 -1.930000e+08 -4.615385e+05                8                  82
min_seg_size_forward           33 -5.368707e+08 -1.000000e+00                4                  33
   Fwd Header Length           33 -3.221223e+10 -1.100000e+01               26                  33
   Bwd Header Length           21 -1

In [7]:
# 6. Semantic families and duration/rate dependency diagnostics (descriptive only).
families={
 "duration_IAT":["Flow Duration","Flow IAT Mean","Flow IAT Std","Flow IAT Max","Flow IAT Min","Fwd IAT Total","Fwd IAT Mean","Fwd IAT Std","Fwd IAT Max","Fwd IAT Min","Bwd IAT Total","Bwd IAT Mean","Bwd IAT Std","Bwd IAT Max","Bwd IAT Min"],
 "rates":["Flow Bytes/s","Flow Packets/s","Fwd Packets/s","Bwd Packets/s"],
 "headers_segments":["Fwd Header Length","Bwd Header Length","min_seg_size_forward"],
 "packet_counts_lengths":["Total Fwd Packets","Total Backward Packets","Total Length of Fwd Packets","Total Length of Bwd Packets","Fwd Packet Length Max","Fwd Packet Length Min","Fwd Packet Length Mean","Fwd Packet Length Std","Bwd Packet Length Max","Bwd Packet Length Min","Bwd Packet Length Mean","Bwd Packet Length Std","Min Packet Length","Max Packet Length","Packet Length Mean","Packet Length Std","Packet Length Variance"],
 "flags_ratios":["Fwd PSH Flags","Bwd PSH Flags","Fwd URG Flags","Bwd URG Flags","FIN Flag Count","SYN Flag Count","RST Flag Count","PSH Flag Count","ACK Flag Count","URG Flag Count","CWE Flag Count","ECE Flag Count","Down/Up Ratio","Average Packet Size","Avg Fwd Segment Size","Avg Bwd Segment Size"]}
fmap={f:k for k,vals in families.items() for f in vals}
per_feature["semantic_family"]=per_feature.feature.map(fmap).fillna("unclassified")
family_summary=per_feature.groupby("semantic_family").agg(features=("feature","count"),occurrences=("occurrences","sum"),feature_row_counts_sum=("unique_source_rows","sum")).reset_index()
raw_idx=raw.set_index(keys)
dep=[]
for key in neg[keys].drop_duplicates().itertuples(index=False,name=None):
    r=raw_idx.loc[key]
    vals={c:pd.to_numeric(pd.Series([r.get(c)]),errors="coerce").iloc[0] for c in ["Flow Duration","Flow Bytes/s","Flow Packets/s"]}
    dep.append({"source_file":key[0],"source_row":int(key[1]),**vals,
      "negative_duration":bool(pd.notna(vals["Flow Duration"]) and vals["Flow Duration"]<0),
      "negative_bytes_rate":bool(pd.notna(vals["Flow Bytes/s"]) and vals["Flow Bytes/s"]<0),
      "negative_packets_rate":bool(pd.notna(vals["Flow Packets/s"]) and vals["Flow Packets/s"]<0)})
dependency=pd.DataFrame(dep)
print(family_summary.to_string(index=False))
print("Negative-duration rows:",int(dependency.negative_duration.sum()))
print("Also negative Flow Bytes/s:",int((dependency.negative_duration & dependency.negative_bytes_rate).sum()))
print("Also negative Flow Packets/s:",int((dependency.negative_duration & dependency.negative_packets_rate).sum()))
print("These are associations, not proof of packet ordering or implementation cause.")

 semantic_family  features  occurrences  feature_row_counts_sum
    duration_IAT         5         2925                    2925
headers_segments         3           87                      87
           rates         2          192                     192
Negative-duration rows: 110
Also negative Flow Bytes/s: 82
Also negative Flow Packets/s: 110
These are associations, not proof of packet ordering or implementation cause.


In [8]:
# 7. Export all audit tables and a cautious, reproducible report.
def save(df,name): df.to_csv(OUT/name,index=False,encoding="utf-8")
save(pd.DataFrame(verification),"source_file_verification.csv")
save(g,"conflict_group_detail.csv")
save(label_combos,"conflict_label_combinations.csv")
save(label_source,"conflict_label_source_split.csv")
save(label_source_matrix.reset_index(),"conflict_label_by_source_matrix.csv")
save(label_split_matrix.reset_index(),"conflict_label_by_split_matrix.csv")
save(largest_profile,"largest_conflict_group_feature_profile.csv")
save(largest_meta,"largest_conflict_group_raw_metadata.csv")
save(raw_conf[keys+["feature_hash","Label_audit","split"]+meta_cols].drop_duplicates(),"conflict_raw_provenance.csv")
if not time_profile.empty: save(time_profile,"conflict_timestamp_profile.csv")
if not tuple_profile.empty: save(tuple_profile,"conflict_five_tuple_profile.csv")
save(per_feature,"negative_values_by_feature.csv")
save(cooccur.reset_index(),"negative_feature_cooccurrence.csv")
save(patterns,"negative_row_patterns.csv")
save(family_summary,"negative_semantic_family_summary.csv")
save(dependency,"negative_duration_rate_dependency.csv")
save(neg_raw[keys+["feature","value","Label_audit","split"]].drop_duplicates(),"negative_raw_provenance.csv")
result={
 "status":"completed","repository_head":HEAD,"dataset_fingerprint":split["dataset_fingerprint"],"split_fingerprint":split["split_fingerprint"],
 "verified_source_files":len(verification),"all_source_checksums_pass":all(x["status"]=="PASS" for x in verification),
 "conflict_groups":len(g),"conflict_rows":len(conf),"label_combinations":len(label_combos),
 "largest_group_rows":len(largest),"largest_group_labels":sorted(largest.Label.unique().tolist()),
 "largest_group_source_counts":{str(k):int(v) for k,v in largest.source_file.value_counts().items()},
 "largest_group_exact_distinct_feature_vectors":int(largest[features].drop_duplicates().shape[0]),
 "conflict_raw_rows_recovered":len(raw_conf),"raw_conflict_labels_match":True,
 "negative_occurrences":len(neg),"negative_unique_source_rows":int(neg[keys].drop_duplicates().shape[0]),
 "negative_features":sorted(neg.feature.unique().tolist()),"negative_raw_values_match":True,
 "raw_timestamp_available":"Timestamp" in raw.columns,"five_tuple_available":not tuple_profile.empty,
 "test_rows_analyzed":False,"test_predictions_performed":False,"test_metrics_computed":False,
 "training_performed":False,"labels_modified":False,"source_data_modified":False,"checkpoints_modified":False,"model_lock_modified":False,
 "interpretation":"CSV-level evidence only; PCAP, packet ordering, attack schedule and exact CICFlowMeter build are needed for causal adjudication."
}
(OUT/"investigation_summary.json").write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding="utf-8")
report=f"""# FL-06 Comprehensive Data Quality Investigation v3

## Integrity
- Repository HEAD: {HEAD}
- Raw source checksums: {len(verification)}/8 PASS
- Dataset fingerprint: {split['dataset_fingerprint']}
- Split fingerprint: {split['split_fingerprint']}
- Upstream v1-v2 comparison: PASS
- Test-specific analysis/prediction/metrics: NOT PERFORMED

## Label conflicts
- Mixed-label groups: {len(g)}
- Rows involved: {len(conf)}
- Distinct label combinations: {len(label_combos)}
- Largest group: {len(largest)} rows
- Largest group labels: {', '.join(sorted(largest.Label.unique().tolist()))}
- Exact distinct 77-feature vectors in largest group: {largest[features].drop_duplicates().shape[0]}
- Raw provenance rows recovered and label-matched: {len(raw_conf)}
- Largest-group source files: {largest.source_file.value_counts().to_dict()}

The CSV establishes inconsistent labels for identical feature vectors, but cannot decide which label is correct. Timestamp and 5-tuple metadata are supporting clues only, not packet-level ground truth.

## Negative values
- Feature-value occurrences: {len(neg)}
- Unique affected source rows: {neg[keys].drop_duplicates().shape[0]}
- Features: {', '.join(sorted(neg.feature.unique().tolist()))}
- Every selected negative value matches the original raw CSV: YES

Feature family, per-feature, per-source, per-label, row co-occurrence and duration/rate diagnostics are exported. Co-occurrence does not prove a CICFlowMeter defect.

## Limits and decision
1. Do not automatically relabel, zero, or delete these records.
2. Exact vector plus mixed labels proves label inconsistency, not the correct ground truth.
3. Negative duration/IAT/rate patterns may be compatible with timestamp/order or flow-construction issues, but this CSV audit cannot prove that cause.
4. Header/segment anomalies require exact feature formula and exporter-version evidence.
5. PCAP and attack-schedule alignment are needed for forensic label adjudication.
6. Exact feature-hash isolation does not establish day/session independence.
7. Preserve v1.3 unchanged; any cleaning policy must create a new immutable dataset version, predeclared before model training.

## Safety
No Test analysis, training, predictions, labels/features/source files/checkpoints/model locks were modified.
"""
(OUT/"INVESTIGATION_REPORT.md").write_text(report,encoding="utf-8")
print(json.dumps(result,indent=2,ensure_ascii=False))
print("Output files:")
for p in sorted(OUT.iterdir()): print(p.name,p.stat().st_size)

{
  "status": "completed",
  "repository_head": "775f2191d3466af15cd3073949d8058a116bcb78",
  "dataset_fingerprint": "831262a47b21af777caf0914d5acb85a7005822267b33a0caf86754d6c380d58",
  "split_fingerprint": "b67f17d162df41840629b22fa258b7e843cf54541ccfd6fff5910f325b70b812",
  "verified_source_files": 8,
  "all_source_checksums_pass": true,
  "conflict_groups": 614,
  "conflict_rows": 6342,
  "label_combinations": 4,
  "largest_group_rows": 3362,
  "largest_group_labels": [
    "BENIGN",
    "DoS Hulk"
  ],
  "largest_group_source_counts": {
    "Wednesday-workingHours.pcap_ISCX.csv": 3360,
    "Tuesday-WorkingHours.pcap_ISCX.csv": 1,
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv": 1
  },
  "largest_group_exact_distinct_feature_vectors": 1,
  "conflict_raw_rows_recovered": 6342,
  "raw_conflict_labels_match": true,
  "negative_occurrences": 3204,
  "negative_unique_source_rows": 2614,
  "negative_features": [
    "Bwd Header Length",
    "Flow Bytes/s",
    "Flow Duration",

## تفسیر و معیار تصمیم‌گیری

این Notebook با موفقیت نشان می‌دهد که ردیف‌های منتخب در Train/Validation از CSVهای اصلی بازیابی شده‌اند و با خروجی Audit v2 تطابق دارند. اما این بررسی به‌تنهایی علت ریشه‌ای را در سطح بسته شبکه اثبات نمی‌کند.

برای داوری قطعی موارد زیر لازم‌اند:
1. PCAP اصلی، برنامه زمانی حملات، نقش مبدأ/مقصد و بازسازی مستقل Flow.
2. ترتیب Timestamp بسته‌ها و نسخه دقیق CICFlowMeter که CSV منتشرشده را تولید کرده است.
3. بررسی کد محاسبه Header و Segment Size و در صورت امکان استخراج مجدد ویژگی‌ها.
4. پروتکل ارزیابی مستقل بر اساس روز یا Session برای سنجش تعمیم‌پذیری.

تصمیم پیشنهادی: نسخه v1.3 را به‌عنوان خط مبنای تاریخی دست‌نخورده نگه دارید. برای هر سیاست اصلاحی، Dataset نسخه‌دار و immutable جدید بسازید. Test نباید برای انتخاب سیاست پاک‌سازی استفاده شود.

In [9]:
# 8. OPTIONAL: publish only this notebook's generated outputs.
# Requires GitHub Contents Read/Write. Review INVESTIGATION_REPORT.md first.
allowed={p.name for p in OUT.iterdir() if p.is_file()}
assert "INVESTIGATION_REPORT.md" in allowed and "investigation_summary.json" in allowed
assert all((OUT/n).stat().st_size<=10*1024*1024 for n in allowed)
status=git(["status","--porcelain","--untracked-files=all"],cwd=ROOT).stdout.splitlines()
expected={str((OUT/n).relative_to(ROOT)) for n in allowed}
unexpected=[x for x in status if not (len(x)>=4 and x[:2]=="??" and x[3:] in expected)]
if unexpected: raise RuntimeError("Unexpected repository changes; refusing push: "+" | ".join(unexpected))
git(["fetch","origin","main"],cwd=ROOT)
local=git(["rev-parse","HEAD"],cwd=ROOT).stdout.strip(); remote=git(["rev-parse","origin/main"],cwd=ROOT).stdout.strip()
if local!=remote:
    anc=git(["merge-base","--is-ancestor","HEAD","origin/main"],cwd=ROOT,check=False)
    if anc.returncode: raise RuntimeError("Remote diverged; no files staged.")
    git(["merge","--ff-only","origin/main"],cwd=ROOT)
git(["add","--",str(OUT.relative_to(ROOT))],cwd=ROOT)
staged=set(git(["diff","--cached","--name-only"],cwd=ROOT).stdout.splitlines())
if staged!=expected: raise RuntimeError("Staged paths differ from strict allowlist.")
git(["config","user.name","FL-06 Data Quality Audit Bot"],cwd=ROOT)
git(["config","user.email","fl06-data-quality-audit-bot@users.noreply.github.com"],cwd=ROOT)
git(["commit","-m","Add comprehensive FL-06 data quality investigation v3"],cwd=ROOT)
new=git(["rev-parse","HEAD"],cwd=ROOT).stdout.strip()
git(["push","origin","HEAD:main"],cwd=ROOT); git(["fetch","origin","main"],cwd=ROOT)
assert git(["rev-parse","origin/main"],cwd=ROOT).stdout.strip()==new
print("GITHUB PUBLICATION VERIFIED",new)
print(sorted(allowed))

GITHUB PUBLICATION VERIFIED b53b923258d0b0a94e86635f1d994c640ef3d989
['INVESTIGATION_REPORT.md', 'conflict_group_detail.csv', 'conflict_label_by_source_matrix.csv', 'conflict_label_by_split_matrix.csv', 'conflict_label_combinations.csv', 'conflict_label_source_split.csv', 'conflict_raw_provenance.csv', 'investigation_summary.json', 'largest_conflict_group_feature_profile.csv', 'largest_conflict_group_raw_metadata.csv', 'negative_duration_rate_dependency.csv', 'negative_feature_cooccurrence.csv', 'negative_raw_provenance.csv', 'negative_row_patterns.csv', 'negative_semantic_family_summary.csv', 'negative_values_by_feature.csv', 'source_file_verification.csv']
